# Team-perspective rewrites of game texts

Each game text (NBA.com written report or recap-video transcript) becomes three
texts:

- **raw** — title and text as stored;
- **home_target** — the home team is `Target Team`, the away team `Opponent Team`;
- **away_target** — the other way round.

Each rewrite opens with a sentence saying which team each label stands for,
then the rewritten title and text. The players listed for the game (box score
and inactive list, from the database) get a note of their side,
"Tyler Herro (Target Team player)", ruled by `LABEL_RULE`:

- `"any"` — no note in a sentence that already contains either team label;
- `"own"` — no note when the sentence contains the label of the player's team;
- `"never"` — a note in every sentence (first mention of each player).

The rewrite is rule-based (`lab/text_perspective/team_perspective.py`): an alias
table per team, only the two teams of the game are replaced, other teams are
left as written. The perspective versions also drop the AP dateline and footer.

In the raw text, **blue** marks the home team, **orange** the away team, grey
underline a team that did not play (left as written) and grey text a city
both teams share (Los Angeles in a Lakers–Clippers game), also left as written.

In [1]:
import html
import re
import sys
from collections import Counter
from pathlib import Path

from IPython.display import HTML, display

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from lab.text_perspective.documents import (  # noqa: E402
    load_game_players,
    s3_client,
    sample_documents,
)
from lab.text_perspective.players import Player, tag_players  # noqa: E402
from lab.text_perspective.team_perspective import (  # noqa: E402
    OPPONENT_LABEL,
    TARGET_LABEL,
    compose,
    document_perspectives,
    replace_teams,
)

In [2]:
PER_SOURCE = 2  # documents drawn from each source
SOURCES = ("report", "transcript")
SEASON = None  # e.g. "2025-26"; None = any season
SEED = None  # an int makes the draw reproducible

client = s3_client()
documents = sample_documents(
    client, per_source=PER_SOURCE, sources=SOURCES, season=SEASON, seed=SEED
)
players = load_game_players(doc["game_id"] for doc in documents)
for doc in documents:
    print(
        f"{doc['source']:<10} {doc['game_id']}  {doc['date']}  "
        f"{doc['away']} @ {doc['home']}  {doc['away_score']}-{doc['home_score']}  "
        f"{doc['word_count']} words  {len(players[doc['game_id']])} players"
    )

/home/panchojasen/Projects/NBA_over_under_predictor/src/nba_ou/postgre_db/row_order.py:116: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


Loaded 111 player records from database
Loaded 23 injury records from database
report     0042400312  2025-05-22  Timberwolves @ Thunder  103-118  460 words  30 players
report     0042100145  2022-04-26  Pelicans @ Suns  97-112  828 words  30 players
transcript 0022500973  2026-03-14  Hornets @ Spurs  102-115  352 words  36 players
transcript 0022500961  2026-03-13  Grizzlies @ Pistons  110-126  321 words  38 players


/home/panchojasen/Projects/NBA_over_under_predictor/src/nba_ou/postgre_db/row_order.py:116: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


In [3]:
COLORS = {"home": "#cfe2ff", "away": "#ffe0b2", "target": "#cfe2ff", "opponent": "#ffe0b2"}


def mark(text: str, color: str) -> str:
    return f'<mark style="background:{color};padding:0 2px">{html.escape(text)}</mark>'


def highlight_raw(text: str, home: str, away: str) -> str:
    # Spans are found with the home team as target: target = home, opponent = away.
    rewrite = replace_teams(text, home, away)
    side = {"target": "home", "opponent": "away"}
    parts, last = [], 0
    for m in rewrite.mentions:
        parts.append(html.escape(text[last : m.start]))
        if m.role in side:
            parts.append(mark(m.surface, COLORS[side[m.role]]))
        elif m.role == "other":
            parts.append(f'<u style="text-decoration-color:#999">{html.escape(m.surface)}</u>')
        else:
            parts.append(f'<span style="color:#888">{html.escape(m.surface)}</span>')
        last = m.end
    parts.append(html.escape(text[last:]))
    return "".join(parts)


def highlight_labels(text: str) -> str:
    out = html.escape(text)
    out = re.sub(r"\((?:Target|Opponent) Team player\)", r"<i>\g<0></i>", out)
    out = out.replace(TARGET_LABEL, mark(TARGET_LABEL, COLORS["target"]))
    return out.replace(OPPONENT_LABEL, mark(OPPONENT_LABEL, COLORS["opponent"]))


def mention_summary(text: str, home: str, away: str) -> str:
    counts = Counter(
        (m.role, m.surface) for m in replace_teams(text, home, away).mentions
    )
    name = {"target": home, "opponent": away, "other": "other teams", "shared": "shared city (kept)"}
    rows = []
    for role in ("target", "opponent", "other", "shared"):
        found = [f"{s} ×{n}" for (r, s), n in counts.most_common() if r == role]
        if found:
            rows.append(f"<b>{html.escape(name[role])}</b>: {html.escape(', '.join(found))}")
    return "<br>".join(rows)


def player_summary(raw: str, doc: dict, roster: list[Player], label_rule: str) -> str:
    # Outcomes from the home team's view; the away view mirrors them.
    team_text = replace_teams(raw, doc["home"], doc["away"]).text
    tagging = tag_players(
        team_text,
        roster,
        doc["home"],
        doc["away"],
        target_label=TARGET_LABEL,
        opponent_label=OPPONENT_LABEL,
        label_rule=label_rule,
    )
    outcomes = Counter(m.outcome for m in tagging.mentions)
    others = sorted({m.surface for m in tagging.mentions if m.outcome == "other_person"})
    line = ", ".join(f"{k} ×{n}" for k, n in outcomes.most_common())
    if others:
        line += f" — taken as somebody else: {', '.join(others)}"
    return f"<b>players</b> ({len(roster)} listed): {html.escape(line)}"


def show(doc: dict, roster: list[Player] | None = None, label_rule: str = "any") -> None:
    roster = roster or []
    texts = document_perspectives(doc, players=roster, label_rule=label_rule)
    raw = compose(doc.get("title") or "", doc["text"])
    box = "white-space:pre-wrap;font-size:13px;line-height:1.45;padding:8px;border:1px solid #ddd;vertical-align:top"
    header = (
        f"<h3>{html.escape(doc['source'])} · {doc['game_id']} · {doc['date']} · "
        f"{html.escape(doc['away'])} @ {html.escape(doc['home'])} "
        f"{doc['away_score']}-{doc['home_score']}</h3>"
        f"<p style='font-size:12px'>{mention_summary(raw, doc['home'], doc['away'])}"
        f"<br>{player_summary(raw, doc, roster, label_rule)}</p>"
    )
    table = (
        "<table style='table-layout:fixed;width:100%'><tr>"
        f"<th>raw</th><th>home_target ({html.escape(doc['home'])})</th>"
        f"<th>away_target ({html.escape(doc['away'])})</th></tr><tr>"
        f"<td style='{box}'>{highlight_raw(raw, doc['home'], doc['away'])}</td>"
        f"<td style='{box}'>{highlight_labels(texts['home_target'])}</td>"
        f"<td style='{box}'>{highlight_labels(texts['away_target'])}</td>"
        "</tr></table>"
    )
    display(HTML(header + table))

In [4]:
LABEL_RULE = "any"  # "any" | "own" | "never"

for doc in documents:
    show(doc, players[doc["game_id"]], LABEL_RULE)

raw,home_target (Suns),away_target (Pelicans)


raw,home_target (Spurs),away_target (Hornets)
"Game Recap: Spurs 115, Hornets 102 - If you're a big game hunt, look no further than the Spurs and Hornets. Spurs winners of 16 of 18, three and a half back of OKC. Hornets 1-2 straight, winners of 8 of 10, sit 10th in the East in the play-in. Check who's playing, it's LaMelo Ball. Maybe he's not playing the assist from Moussa Diabate. Ball, 17 points in the game. Then Stephon Castle finds the alien. Victor Wembinyama rings up the triple. On the break, Castle, I hope he gets credited with an assist for the... unconventional play to keldon johnson the alley oops and now let's all put ourselves in grant williams shoes who's got a full foot on grant williams williams trying to body him up he felt that spun away and then the left-hand hammer wembie 11 points and two blocks in the first quarter alone then you see the triple from colby white off the knipple pass hornets end the first - busy spinning making the bucket and then Miles Bridges with the turnaround fade away that goes Wemby pushes the ball up the floor playing point center finds De'Aaron Fox for the triple Fox had 17 points in the game 57 to 46 halftime over back to the action Bridges kicks it out to Knipple catch and shoot triples good top play Canada here is Stephon Castles doing pull-ups calisthenics with the two-hand finish through the defense miles bridges with the fly by the waves now it's time for knipple to cut and finish on the body the strength of the rook he had 17 points just in the third quarter 20 in the game then wemby drives and throws a perfect alley-oop to a rolling luke cornet cornets win the third quarter by five they trail by eight heading into the fourth And now it's time for the Area 51 connection to go off. As you see, Wemby getting fed from his boy, Stephon Castle. And Stephon Castle go in full dunk contest mode with a 360. Almost put his arm in the rim backwards. Final score, Spurs win it, 115-102.","In this text, Target Team is the San Antonio Spurs (home team) and Opponent Team is the Charlotte Hornets (visiting team). Game Recap: Target Team 115, Opponent Team 102 If you're a big game hunt, look no further than the Target Team and Opponent Team. Target Team winners of 16 of 18, three and a half back of OKC. Opponent Team 1-2 straight, winners of 8 of 10, sit 10th in the East in the play-in. Check who's playing, it's LaMelo Ball (Opponent Team player). Maybe he's not playing the assist from Moussa Diabate (Opponent Team player). Ball (Opponent Team player), 17 points in the game. Then Stephon Castle (Target Team player) finds the alien. Victor Wembinyama (Target Team player) rings up the triple. On the break, Castle (Target Team player), I hope he gets credited with an assist for the... unconventional play to keldon johnson the alley oops and now let's all put ourselves in grant williams shoes who's got a full foot on grant williams williams trying to body him up he felt that spun away and then the left-hand hammer wembie 11 points and two blocks in the first quarter alone then you see the triple from colby white off the knipple pass hornets end the first - busy spinning making the bucket and then Miles Bridges (Opponent Team player) with the turnaround fade away that goes Wemby pushes the ball up the floor playing point center finds De'Aaron Fox (Target Team player) for the triple Fox had 17 points in the game 57 to 46 halftime over back to the action Bridges kicks it out to Knipple catch and shoot triples good top play Canada here is Stephon Castles (Target Team player) doing pull-ups calisthenics with the two-hand finish through the defense miles bridges with the fly by the waves now it's time for knipple to cut and finish on the body the strength of the rook he had 17 points just in the third quarter 20 in the game then wemby drives and throws a perfect alley-oop to a rolling luke cornet cornets win the third quarter by five they trail by eight heading into the fourth And now it's time for the 

raw,home_target (Pistons),away_target (Grizzlies)
"Game Recap: Pistons 126, Grizzlies 110 - The Grizzlies taking on the top seed in the East, the Detroit Pistons, and Cade Cunningham, who was presented with the Eastern Conference Player of the Month Award for February. And Cunningham with another big game, mostly did it, dishing it off here to Javante Green, who hits the corner three. Cunningham entering the day number two in the NBA in assists per game. Javante Small the other way for Memphis, the pump fake step back three. Small at a team high, 23 points off the bench for Memphis. Isaiah Stewart gets it to Cunningham, who finishes with the jam, not just dishing it off all the time. Cunningham scored seven in the opening half. How about Taj Gibson, 40 years old, debuting with his eighth NBA team, hits the three there. The Grizzlies hanging around after one quarter. - pull up three drills at Grizzlies leading the Pistons by five at this point and Cunningham gets it to Duren for the alley oop Pistons trying to take back control up five go to the second half Cunningham finds Marcus Sasser in the corner what a look Sasser with 16 points on the ninth and then Cade with another great assist look at the vision gets it to Duren Can't be stopped down low. The and one layup against two defenders. Pistons go on a 21-7 run. Now we pick it up in the fourth with the Pistons up 20. Jerome does hit a three here. 21 points on the night for Memphis, but this was about Detroit. How about Duren getting it done on the boards, fighting at least three Grizzlies there. Just a bunch of white jerseys around him. Duren, 30 points, 13 rebounds in the game. Cunningham, he's... Scoring against a bunch of white jerseys as well. Look at that nice one-hand flush. Cunningham, 17 points, 15 assists. Pistons roll at home to win their third straight.","In this text, Target Team is the Detroit Pistons (home team) and Opponent Team is the Memphis Grizzlies (visiting team). Game Recap: Target Team 126, Opponent Team 110 The Opponent Team taking on the top seed in the East, the Target Team, and Cade Cunningham, who was presented with the Eastern Conference Player of the Month Award for February. And Cunningham (Target Team player) with another big game, mostly did it, dishing it off here to Javante Green (Target Team player), who hits the corner three. Cunningham (Target Team player) entering the day number two in the NBA in assists per game. Javante Small the other way for Opponent Team, the pump fake step back three. Small at a team high, 23 points off the bench for Opponent Team. Isaiah Stewart (Target Team player) gets it to Cunningham (Target Team player), who finishes with the jam, not just dishing it off all the time. Cunningham (Target Team player) scored seven in the opening half. How about Taj Gibson (Opponent Team player), 40 years old, debuting with his eighth NBA team, hits the three there. The Opponent Team hanging around after one quarter. - pull up three drills at Opponent Team leading the Target Team by five at this point and Cunningham gets it to Duren for the alley oop Target Team trying to take back control up five go to the second half Cunningham finds Marcus Sasser in the corner what a look Sasser with 16 points on the ninth and then Cade with another great assist look at the vision gets it to Duren Can't be stopped down low. The and one layup against two defenders. Target Team go on a 21-7 run. Now we pick it up in the fourth with the Target Team up 20. Jerome (Opponent Team player) does hit a three here. 21 points on the night for Opponent Team, but this was about Target Team. How about Duren getting it done on the boards, fighting at least three Opponent Team there. Just a bunch of white jerseys around him. Duren (Target Team player), 30 points, 13 rebounds in the game. Cunningham (Target Team player), he's... Scoring against a bunch of white jerseys as well. Look at that nice one-hand flush. Cunningham (Target Team player), 17 points, 15 as

## Try a text of your own

`home` / `away` are the schedule nicknames stored with every document
(`"Lakers"`, `"Timberwolves"`, `"Trail Blazers"`, `"76ers"`, ...); a player's
`team` uses the same names.

In [5]:
custom = {
    "source": "custom",
    "game_id": "-",
    "date": "-",
    "home": "Lakers",
    "away": "Timberwolves",
    "home_score": 0,
    "away_score": 0,
    "title": "Doncic scores 38 as Lakers beat Timberwolves 118-110",
    "text": (
        "LOS ANGELES (AP) — Luka Doncic scored 38 points and the Los Angeles Lakers "
        "beat the Minnesota Timberwolves 118-110. The Wolves trailed by 12 at the half "
        "before Anthony Edwards brought Minnesota within two. Doncic and Reaves combined "
        "for 61, while Edwards finished with 31. Rudy Gobert (left ankle) did not play. "
        "Kobe Bryant's statue stood outside. L.A. closed on a 9-0 run. "
        "The Lakers host the LA Clippers on Friday; the T-Wolves visit Denver."
    ),
}
custom_players = [
    Player("Luka", "Dončić", "Lakers"),
    Player("Austin", "Reaves", "Lakers"),
    Player("Kobe", "Bufkin", "Lakers"),
    Player("Anthony", "Edwards", "Timberwolves"),
    Player("Rudy", "Gobert", "Timberwolves"),
]
show(custom, custom_players, LABEL_RULE)

raw,home_target (Lakers),away_target (Timberwolves)
"Doncic scores 38 as Lakers beat Timberwolves 118-110 LOS ANGELES (AP) — Luka Doncic scored 38 points and the Los Angeles Lakers beat the Minnesota Timberwolves 118-110. The Wolves trailed by 12 at the half before Anthony Edwards brought Minnesota within two. Doncic and Reaves combined for 61, while Edwards finished with 31. Rudy Gobert (left ankle) did not play. Kobe Bryant's statue stood outside. L.A. closed on a 9-0 run. The Lakers host the LA Clippers on Friday; the T-Wolves visit Denver.","In this text, Target Team is the Los Angeles Lakers (home team) and Opponent Team is the Minnesota Timberwolves (visiting team). Doncic scores 38 as Target Team beat Opponent Team 118-110 Luka Doncic scored 38 points and the Target Team beat the Opponent Team 118-110. The Opponent Team trailed by 12 at the half before Anthony Edwards brought Opponent Team within two. Doncic (Target Team player) and Reaves (Target Team player) combined for 61, while Edwards (Opponent Team player) finished with 31. Rudy Gobert (Opponent Team player) (left ankle) did not play. Kobe Bryant's statue stood outside. Target Team closed on a 9-0 run. The Target Team host the LA Clippers on Friday; the Opponent Team visit Denver.","In this text, Target Team is the Minnesota Timberwolves (visiting team) and Opponent Team is the Los Angeles Lakers (home team). Doncic scores 38 as Opponent Team beat Target Team 118-110 Luka Doncic scored 38 points and the Opponent Team beat the Target Team 118-110. The Target Team trailed by 12 at the half before Anthony Edwards brought Target Team within two. Doncic (Opponent Team player) and Reaves (Opponent Team player) combined for 61, while Edwards (Target Team player) finished with 31. Rudy Gobert (Target Team player) (left ankle) did not play. Kobe Bryant's statue stood outside. Opponent Team closed on a 9-0 run. The Opponent Team host the LA Clippers on Friday; the Target Team visit Denver."
